In [8]:
import timm
import torch
import pandas as pd
from pprint import pprint
from pathlib import Path

from redundant_features import SCALES

In [2]:
imagenet_dfs = []
metric = 'accuracy'

for n in [10, 20, 50, 100]:
    vit_df = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/timm_mae_base/knn_scores_5way_ImageNet{n:03d}.csv").drop('Unnamed: 0', axis=1)
    vit_df = vit_df[vit_df.dim==768].reset_index(drop=True)
    vit_df = vit_df.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
    vit_df = vit_df[vit_df.index.isin(SCALES)]
    vit_df['n'] = n
    vit_df['max_val'] = max(vit_df[metric].values)
    vit_df['rel_ret_rate'] = vit_df[metric]/vit_df['max_val']
    vit_df['rel_ret_rate_p'] = vit_df['rel_ret_rate']*100
    vit_df['rel_ret_rate_p'] = vit_df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
    imagenet_dfs.append(vit_df.reset_index())  # reset so 'scale' becomes a column again

df_imagenet = pd.concat(imagenet_dfs, ignore_index=True)
pivot = df_imagenet.pivot(index='n', columns='scale', values='rel_ret_rate_p')
print(pivot.to_markdown())

|   n | 0.001   | 0.01   | 0.1   | 0.25   | 0.5   | 1.0    |
|----:|:--------|:-------|:------|:-------|:------|:-------|
|  10 | 33.1%   | 33.4%  | 38.0% | 46.4%  | 55.0% | 100.0% |
|  20 | 24.5%   | 24.8%  | 32.7% | 42.8%  | 48.1% | 100.0% |
|  50 | 13.9%   | 14.6%  | 22.1% | 31.8%  | 41.8% | 100.0% |
| 100 | 10.1%   | 10.5%  | 14.6% | 22.6%  | 33.3% | 100.0% |


In [3]:
imagenet_dfs = []
metric = 'accuracy'

for n in [10, 20, 50, 100]:
    vit_df = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/timm_dinoV2_base/knn_scores_5way_ImageNet{n:03d}.csv").drop('Unnamed: 0', axis=1)
    vit_df = vit_df[vit_df.dim==768].reset_index(drop=True)
    vit_df = vit_df.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
    vit_df = vit_df[vit_df.index.isin(SCALES)]
    vit_df['n'] = n
    vit_df['max_val'] = max(vit_df[metric].values)
    vit_df['rel_ret_rate'] = vit_df[metric]/vit_df['max_val']
    vit_df['rel_ret_rate_p'] = vit_df['rel_ret_rate']*100
    vit_df['rel_ret_rate_p'] = vit_df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
    imagenet_dfs.append(vit_df.reset_index())  # reset so 'scale' becomes a column again

df_imagenet = pd.concat(imagenet_dfs, ignore_index=True)
pivot = df_imagenet.pivot(index='n', columns='scale', values='rel_ret_rate_p')
print(pivot.to_markdown())

|   n | 0.001   | 0.01   | 0.1   | 0.25   | 0.5   | 1.0    |
|----:|:--------|:-------|:------|:-------|:------|:-------|
|  10 | 28.6%   | 28.1%  | 35.5% | 47.2%  | 59.5% | 100.0% |
|  20 | 17.5%   | 17.6%  | 24.9% | 35.4%  | 47.1% | 100.0% |
|  50 | 7.6%    | 7.7%   | 13.0% | 22.2%  | 32.5% | 100.0% |
| 100 | 4.1%    | 4.4%   | 7.6%  | 14.6%  | 22.4% | 100.0% |


In [4]:
import geobench
from tqdm import tqdm
import lightning.pytorch as pl
from geobench_evaluate import get_classification_transform
from operator import itemgetter
from pathlib import Path
import numpy as np

In [5]:
from scripts.terratorch_slimable import make_terratorch_prithvi_dofa_slimable
model = timm.create_model(
    "vit_base_patch16_224.mae",
    pretrained=True,
)
model = make_terratorch_prithvi_dofa_slimable(model)
model_name = "timm_mae_base"

dataset = 'm-bigearthnet'
task = [t for t in geobench.task_iterator(benchmark_name="classification_v1.0") if t.dataset_name == dataset][0]
ds = task.get_dataset(split="train")
geobench_dataset_name = ds.task_specs.dataset_name
from geobench_evaluate import bandorders
bandnames = itemgetter(*[4,3,2])(ds.task_specs.bands_info)
bandnames_param = [bn.name for bn in bandnames]
dm = task.get_pytorch_data_module(
    band_names=bandnames_param,
    train_transform=get_classification_transform(task, bandnames_param, train=True, input_size=224),
    eval_transform=get_classification_transform(task, bandnames_param, train=False, input_size=224),
    batch_size=64,
    num_workers=8,
)

from scripts.redundant_features import extract_features, calculate_result

get_features = False
knn = False

if get_features:
    extract_features(model=model, dm=dm, model_name=model_name, dataset_name=dataset)
if knn:
    calculate_result(model_name=model_name, dataset_name=dataset, max_dim=768, downstream_type='knn')

In [6]:
from scripts.terratorch_slimable import make_terratorch_prithvi_dofa_slimable
model = timm.create_model(
    #"vit_base_patch16_224.mae",
    "vit_base_patch14_dinov2", img_size=224,
    pretrained=True,
)
model = make_terratorch_prithvi_dofa_slimable(model)
model_name = "timm_dinoV2_base"

dataset = 'm-eurosat'
task = [t for t in geobench.task_iterator(benchmark_name="classification_v1.0") if t.dataset_name == dataset][0]
ds = task.get_dataset(split="train")
geobench_dataset_name = ds.task_specs.dataset_name
from geobench_evaluate import bandorders
bandnames = itemgetter(*[4,3,2])(ds.task_specs.bands_info)
bandnames_param = [bn.name for bn in bandnames]
dm = task.get_pytorch_data_module(
    band_names=bandnames_param,
    train_transform=get_classification_transform(task, bandnames_param, train=True, input_size=224),
    eval_transform=get_classification_transform(task, bandnames_param, train=False, input_size=224),
    batch_size=64,
    num_workers=8,
)

from scripts.redundant_features import extract_features, calculate_result

get_features = False
knn = False

if get_features:
    extract_features(model=model, dm=dm, model_name=model_name, dataset_name=dataset)
if knn:
    calculate_result(model_name=model_name, dataset_name=dataset, max_dim=768, downstream_type='knn')

In [7]:
for metric, ds in [('accuracy', 'm-eurosat'), ('mAP', 'm-bigearthnet')]:
    dfs = []
    
    for m in ['prithvi2_300', 'prithvi2_600', 'dofa', 'dofa_large', 'terramind', 'terramind_large']:
        df = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/SlimAttn+SlimFFN/{m}/knn_scores_5way_{ds}.csv").drop('Unnamed: 0', axis=1)
        df = df[df.dim==max(df.dim.values)].reset_index(drop=True)
        df = df.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
        df = df[df.index.isin(SCALES)]
        df['model'] = m
        df['max_val'] = max(df[metric].values)
        df['rel_ret_rate'] = df[metric]/df['max_val']
        df['rel_ret_rate_p'] = df['rel_ret_rate']*100
        df['rel_ret_rate_p'] = df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
        df['metric_p'] = df[metric]*100
        df['metric_p'] = df['metric_p'].apply(lambda x: f"{x:4.1f}%")
        dfs.append(df.reset_index())  # reset so 'scale' becomes a column again
    
    df = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/timm_mae_base/knn_scores_5way_{ds}.csv").drop('Unnamed: 0', axis=1)
    df = df[df.dim==max(df.dim.values)].reset_index(drop=True)
    df = df.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
    df = df[df.index.isin(SCALES)]
    df['model'] = 'CV MAE'
    df['max_val'] = max(df[metric].values)
    df['rel_ret_rate'] = df[metric]/df['max_val']
    df['rel_ret_rate_p'] = df['rel_ret_rate']*100
    df['rel_ret_rate_p'] = df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
    df['metric_p'] = df[metric]*100
    df['metric_p'] = df['metric_p'].apply(lambda x: f"{x:4.1f}%")
    dfs.append(df.reset_index())  # reset so 'scale' becomes a column again
    
    df = pd.concat(dfs, ignore_index=True)
    pivot = df.pivot(index='model', columns='scale', values='metric_p')
    print(f'{ds} ({metric})')
    print(pivot.to_markdown())
    print()
    print(f'{ds}, (Rel. Retention Rate of {metric})')
    pivot = df.pivot(index='model', columns='scale', values='rel_ret_rate_p')
    print(pivot.to_markdown())
    print('\n')

m-eurosat (accuracy)
| model           | 0.001   | 0.01   | 0.1   | 0.25   | 0.5   | 1.0   |
|:----------------|:--------|:-------|:------|:-------|:------|:------|
| CV MAE          | 56.6%   | 57.5%  | 63.4% | 68.9%  | 76.3% | 76.4% |
| dofa            | 71.3%   | 71.4%  | 74.8% | 79.5%  | 83.6% | 82.1% |
| dofa_large      | 74.9%   | 74.8%  | 76.7% | 82.1%  | 86.8% | 91.1% |
| prithvi2_300    | 79.2%   | 79.1%  | 79.9% | 83.3%  | 82.8% | 83.1% |
| prithvi2_600    | 76.9%   | 79.2%  | 82.9% | 81.8%  | 82.0% | 86.4% |
| terramind       | 73.5%   | 73.6%  | 77.8% | 81.4%  | 85.1% | 85.7% |
| terramind_large | 73.9%   | 75.2%  | 81.0% | 84.0%  | 84.8% | 88.9% |

m-eurosat, (Rel. Retention Rate of accuracy)
| model           | 0.001   | 0.01   | 0.1   | 0.25   | 0.5    | 1.0    |
|:----------------|:--------|:-------|:------|:-------|:-------|:-------|
| CV MAE          | 74.1%   | 75.2%  | 83.0% | 90.1%  | 99.9%  | 100.0% |
| dofa            | 85.3%   | 85.4%  | 89.5% | 95.1%  | 100.0% 

In [1]:
import torch
import math

def select_four_layers(features: list[torch.Tensor], img_size: int, patch_size: int) -> torch.Tensor:
    """
    Args:
        features:   List of tensors, each of shape (B, N, D) or (B, D, H_p, W_p)
        img_size:   Input image size (assumes square)
        patch_size: Patch size used by the backbone (assumes square)
    Returns:
        Tensor of shape (B, 4, D, H_p, W_p)
        where H_p = W_p = img_size // patch_size
    """
    n_patches_per_side = img_size // patch_size
    n_patches = n_patches_per_side ** 2
    n = len(features)
    indices = [0, 0, 0, 0] if n == 1 else [round(i * (n - 1) / 3) for i in range(4)]

    first = features[0]
    if first.dim() == 4:
        # Input is (B, D, H_p, W_p)
        stacked = torch.stack([features[i] for i in indices], dim=1)  # (B, 4, D, H_p, W_p)
    else:
        # Input is (B, N, D)
        n_tokens = first.shape[1]
        assert n_tokens in (n_patches, n_patches + 1), (
            f"Unexpected token count {n_tokens}, expected {n_patches} or {n_patches + 1}"
        )
        has_cls = n_tokens == n_patches + 1
        stacked = torch.stack([features[i] for i in indices], dim=1)  # (B, 4, N, D)
        if has_cls:
            stacked = stacked[:, :, 1:, :]  # (B, 4, P*P, D)
        B, four, _, D = stacked.shape
        stacked = stacked.reshape(B, four, n_patches_per_side, n_patches_per_side, D)
        stacked = stacked.permute(0, 1, 4, 2, 3)  # (B, 4, D, H_p, W_p)

    return stacked.contiguous()

In [2]:
BASE_PATH = "/scratch/htc/lhackel/RSFM_redundancy/_features/SlimAttn+SlimFFN"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def get_multiscale_features_and_labels(model, dm, mode='train'):
    assert mode in ['train', 'val', 'test'], f"mode has to be in ['train', 'val', 'test'] but was {mode}"
    pl.seed_everything(42, verbose=False)
    model.to(device)
    model.eval()
    inputs = []
    labels = []
    if isinstance(dm.val_dataloader(), torch.utils.data.DataLoader):
        val_dl = dm.val_dataloader()
    else:
        val_dl = dm.val_dataloader()[0]
    dl = dm.train_dataloader() if mode == 'train' else val_dl if mode == 'val' else dm.test_dataloader()
    print_proc_unknown = True
    #for i, batch in tqdm(enumerate(dl), desc="Extracting features", total=len(dl), leave=False):
    for i, batch in enumerate(dl):
        x = batch['input'].to(device)
        y = batch['label']

        with torch.no_grad():
            features = model(x)
            features = select_four_layers(features, img_size=224, patch_size=ps)

        features = torch.nn.functional.normalize(features, dim=-1)
        features = features.cpu()
        inputs.append(features)
        labels.append(y)

    all_features = torch.cat(inputs, dim=0).cpu().numpy()
    all_labels = torch.cat(labels, dim=0).cpu().numpy()

    return all_features, all_labels

def extract_multiscale_features(model, dm, model_name, dataset_name):
    for scale in tqdm(SCALES, desc=f"Iterating scales"):
        #print(f"{model_name} : {scale}")
        for mode in ['train', 'val', 'test']:
            # sometimes the backbone, sometimes the main model has this attribute
            try:
                model.backbone.set_to_scale(scale)
            except AttributeError:
                model.set_to_scale(scale)
            features, labels = get_multiscale_features_and_labels(model, dm, mode=mode)
            Path(f"{BASE_PATH}/{model_name}/scale_{scale}/").mkdir(exist_ok=True, parents=True)
            np.save(f"{BASE_PATH}/{model_name}/scale_{scale}/features_{dataset_name}_{mode}.npy", features)
            np.save(f"{BASE_PATH}/{model_name}/scale_{scale}/labels_{dataset_name}_{mode}.npy", labels)

## Segmentation

In [4]:
import geobench
from tqdm import tqdm
import lightning.pytorch as pl
from terratorch import BACKBONE_REGISTRY
from terratorch_baselines import evaluated_models
from terratorch_baselines import bandorders as terratorch_baseline_bandorders
from terratorch_slimable import make_terratorch_prithvi_dofa_slimable, make_terratorch_terramind_slimable
from geobench_evaluate import get_segmentation_transform
from operator import itemgetter
from pathlib import Path
import numpy as np
from redundant_features import SCALES

extract_segmentation = False
#dataset = 'm-cashew-plant'
dataset = 'm-SA-crop-type'
if extract_segmentation:
    geobench_task = [t for t in geobench.task_iterator(benchmark_name="segmentation_v1.0") if t.dataset_name == dataset][0]
    geobench_dataset = geobench_task.get_dataset(split="train")
    
    for name_or_path in ['ssl4eo_moco', 'ssl4eo_dino']:
        ps = 14 if name_or_path == 'prithvi2_600' else 16
        batch_size = 64
        
        bandnames = itemgetter(*terratorch_baseline_bandorders[name_or_path]["segmentation_v1.0"][dataset])(
            geobench_dataset.task_specs.bands_info)
        bandnames_param = [bn.name for bn in bandnames]
        if 'Cloud Probability' in bandnames_param:
            # cloud prob does not work, use 12-SWIR as placeholder band -> will be removed in KNN & linear as dublicate
            bandnames_param[bandnames_param.index('Cloud Probability')] = '12 - SWIR'
        
        dm = geobench_task.get_pytorch_data_module(
            band_names=bandnames_param,
            train_transform=get_segmentation_transform(geobench_task, bandnames_param, train=True, input_size=224),
            eval_transform=get_segmentation_transform(geobench_task, bandnames_param, train=False, input_size=224),
            batch_size=batch_size,
            num_workers=16,
        )
        
        version = name_or_path
        backbone = BACKBONE_REGISTRY.build(**evaluated_models[name_or_path]["kwargs"])
        if "prithvi" in name_or_path or "dofa" in name_or_path or "ssl4eo" in name_or_path:
            backbone = make_terratorch_prithvi_dofa_slimable(backbone)
        elif 'terramind' in name_or_path:
            backbone = make_terratorch_terramind_slimable(backbone)
        
        extract_multiscale_features(model=backbone, dm=dm, model_name=name_or_path, dataset_name=dataset)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SegmentationHead(nn.Module):
    """
    Lightweight SegFormer-style MLP head for frozen backbone probing.
    
    Inputs:  (B, 4, C, H_p, W_p)  - pre-extracted multi-scale features
    Outputs: (B, num_classes, H, W) - full resolution segmentation logits
    """
    def __init__(self, in_channels: int, num_classes: int, embed_dim: int = 256, img_size: int = 224, patch_size: int = 16):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size

        # Project each of the 4 layers to common embed_dim
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_channels, embed_dim),
                nn.ReLU(),
            ) for _ in range(4)
        ])

        # Fuse 4 projected features
        self.fuse = nn.Sequential(
            nn.Linear(4 * embed_dim, embed_dim),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 4, C, H_p, W_p)
        B, _, C, H_p, W_p = x.shape

        # Project each layer independently
        projected = []
        for i, proj in enumerate(self.projections):
            xi = x[:, i]                              # (B, C, H_p, W_p)
            xi = xi.permute(0, 2, 3, 1)              # (B, H_p, W_p, C)
            xi = proj(xi)                             # (B, H_p, W_p, embed_dim)
            projected.append(xi)

        # Fuse across layers
        x = torch.cat(projected, dim=-1)             # (B, H_p, W_p, 4*embed_dim)
        x = self.fuse(x)                             # (B, H_p, W_p, embed_dim)
        x = self.classifier(x)                       # (B, H_p, W_p, num_classes)

        # Upsample to full resolution
        x = x.permute(0, 3, 1, 2)                   # (B, num_classes, H_p, W_p)
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)

        return x                                      # (B, num_classes, H, W)

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchmetrics.segmentation import MeanIoU
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score


def train(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    epochs: int = 30,
    lr: float = 0.01,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    """
    Train a segmentation head on pre-extracted features.

    Args:
        model:      SegmentationHead
        features:   (N, 4, C, H_p, W_p)
        labels:     (N, 1, H, W)
        epochs:     Number of training epochs
        lr:         Learning rate for AdamW
        batch_size: Batch size
        device:     Device to train on

    Returns:
        Dict with training history (loss per epoch)
    """
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=255)

    history = {"loss": []}
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)                           # (B, num_classes, H, W)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)

        scheduler.step()
        epoch_loss /= len(dataset)
        history["loss"].append(epoch_loss)
        print(f"  Training epoch {epoch+1:3d}/{epochs} | loss: {epoch_loss:.4f}")

    return history


def test(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 32,
    num_classes: int = 7,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    """
    Evaluate a segmentation head on pre-extracted features.

    Args:
        model:        SegmentationHead
        features:     (N, 4, C, H_p, W_p)
        labels:       (N, 1, H, W)
        batch_size:   Batch size
        num_classes:  Number of semantic classes
        ignore_index: Label value to ignore (e.g. 255 for unlabeled)
        device:       Device to run on

    Returns:
        Dict with mIoU, per-class IoU, accuracy, and F1
    """
    print("  Testing")
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

    model = model.to(device)
    model.eval()

    miou_metric = MeanIoU(num_classes=num_classes).to(device)
    acc_metric = MulticlassAccuracy(num_classes=num_classes, average="micro").to(device)
    f1_metric = MulticlassF1Score(num_classes=num_classes, average="macro").to(device)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                           # (B, num_classes, H, W)
            preds = logits.argmax(dim=1)                # (B, H, W)
            miou_metric.update(preds, y)
            acc_metric.update(preds, y)
            f1_metric.update(preds, y)

    per_class_iou = miou_metric.compute()               # (num_classes,)
    metrics = {
        "mIoU":          per_class_iou.mean().item(),
        "per_class_iou": per_class_iou.cpu().tolist(),
        "accuracy":      acc_metric.compute().item(),
        "macro_f1":      f1_metric.compute().item(),
    }

    print(f"  mIoU: {metrics['mIoU']:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['macro_f1']:.4f}")
    return metrics

In [7]:
import numpy as np
import pandas as pd


model_channel_data = [
    #('prithvi2_300', 1024, 16), 
    #('prithvi2_600', 1280, 14), 
    #('dofa', 768, 16), 
    #('dofa_large', 1024, 16), 
    #('terramind', 768, 16), 
    #('terramind_large', 1024, 16),
    # ('ssl4eo_moco', 384, 16),
    ('ssl4eo_dino', 384, 16),
]


eval_segmentation = True
if eval_segmentation:
    for name, d, p in model_channel_data:
        results = []
        csv_path = f"{BASE_PATH}/{name}/segmentation_scores_probe_{dataset}.csv"
        for scale in SCALES[-2:]:
            print(f"{name:>15} : {scale:5.3f}")
            print('-'*(15+3+5))
            head = SegmentationHead(in_channels=d, num_classes=7 if dataset == 'm-cashew-plant' else 10, embed_dim=256, img_size=224, patch_size=p)
    
            print("Loading training data")
            train_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_{dataset}_train.npy")
            train_labels = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_{dataset}_train.npy")
            train(model=head, features=train_features, labels=train_labels, epochs=30, lr=0.01)
            
            print("Loading test data")
            test_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_{dataset}_test.npy")
            test_labels = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_{dataset}_test.npy")
            metrics = test(model=head, features=test_features, labels=test_labels, num_classes=7 if dataset == 'm-cashew-plant' else 10)
        
            print("Saving results")
            results.append({
                "model": name,
                "scale": scale,
                "dataset": dataset,
                "mIoU": metrics["mIoU"],
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
            })
    
            # Write header only if file doesn't exist yet
            df_row = pd.DataFrame(results[-1:])
            df_row.to_csv(csv_path, mode='a', index=False, header=not Path(csv_path).exists())
            print()
    print("Segmentation complete")

    ssl4eo_dino : 0.800
-----------------------
Loading training data
  Training epoch   1/30 | loss: 1.5525
  Training epoch   2/30 | loss: 1.3081
  Training epoch   3/30 | loss: 1.2521
  Training epoch   4/30 | loss: 1.2132
  Training epoch   5/30 | loss: 1.1760
  Training epoch   6/30 | loss: 1.1499
  Training epoch   7/30 | loss: 1.1214
  Training epoch   8/30 | loss: 1.1120
  Training epoch   9/30 | loss: 1.0875
  Training epoch  10/30 | loss: 1.0614
  Training epoch  11/30 | loss: 1.0522
  Training epoch  12/30 | loss: 1.0281
  Training epoch  13/30 | loss: 1.0064
  Training epoch  14/30 | loss: 0.9782
  Training epoch  15/30 | loss: 0.9643
  Training epoch  16/30 | loss: 0.9445
  Training epoch  17/30 | loss: 0.9226
  Training epoch  18/30 | loss: 0.9045
  Training epoch  19/30 | loss: 0.8820
  Training epoch  20/30 | loss: 0.8624
  Training epoch  21/30 | loss: 0.8419
  Training epoch  22/30 | loss: 0.8267
  Training epoch  23/30 | loss: 0.8084
  Training epoch  24/30 | loss: 0

### Preprocess

In [8]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path

TEST_CITIES = ['brasilia','montpellier','norcia','rio','saclay_w','valencia','dubai','lasvegas','milano','chongqing']
TRAIN_CITIES = ['aguasclaras','bercy','bordeaux','nantes','paris','rennes','saclay_e','abudhabi','cupertino','pisa','beihai','hongkong','beirut','mumbai']
OSCD_BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']


def preprocess_oscd(
    root: str | Path,
    out_root: str | Path,
    crop_size: int = 256,
    train_stride: int = 128,
):
    """
    Crop all OSCD images and labels into fixed-size tiles and save as GeoTIFF.

    Output structure:
        out_root/
            train/
                abudhabi_0_0/
                    img1.tif   (C, crop_size, crop_size)
                    img2.tif
                    label.tif  (crop_size, crop_size)
                abudhabi_0_128/
                    ...
            test/
                brasilia_0_0/
                    ...
    """
    root = Path(root)
    out_root = Path(out_root)
    out_root.mkdir(exist_ok=True)

    img_root   = root / 'Onera Satellite Change Detection dataset - Images'
    train_root = root / 'Onera Satellite Change Detection dataset - Train Labels'
    test_root  = root / 'Onera Satellite Change Detection dataset - Test Labels'

    splits = [
        ('train', TRAIN_CITIES, train_root, train_stride),
        ('test',  TEST_CITIES,  test_root,  crop_size),    # no overlap for test
    ]

    for split, cities, label_root, stride in splits:
        for city in cities:
            print(f"Processing {split}/{city}...")

            # Read all 13 bands for both times (save all bands, select at load time)
            img1 = read_image(img_root / city, '1', OSCD_BANDS)  # (13, H, W)
            img2 = read_image(img_root / city, '2', OSCD_BANDS)  # (13, H, W)
            label = read_label(label_root, city)                  # (H, W)

            H, W = label.shape
            crops = get_crops(H, W, crop_size, stride)

            for r, c in crops:
                crop_dir = out_root / split / f'{city}_{r}_{c}'
                crop_dir.mkdir(parents=True, exist_ok=True)

                _save_tif(img1[:, r:r+crop_size, c:c+crop_size], crop_dir / 'img1.tif')
                _save_tif(img2[:, r:r+crop_size, c:c+crop_size], crop_dir / 'img2.tif')
                _save_tif(label[None, r:r+crop_size, c:c+crop_size], crop_dir / 'label.tif')

def read_image(city_dir: Path, time: str, bands: list[str]) -> np.ndarray:
    """Read selected bands from imgs_{time}_rect, return (C, H, W) float32."""
    rect_dir = city_dir / f'imgs_{time}_rect'
    arrays = []
    for band in bands:
        tif = rect_dir / f'{band}.tif'
        with rasterio.open(tif) as src:
            arrays.append(src.read(1).astype(np.float32))
    return np.stack(arrays, axis=0)   # (C, H, W)

def read_label(label_dir: Path, city: str) -> np.ndarray:
    """Read binary change map, return (H, W) uint8 with values 0/1."""
    tif = label_dir / city / 'cm' / f'{city}-cm.tif'
    with rasterio.open(tif) as src:
        cm = src.read(1)
    return (cm != 1).astype(np.uint8)  # binarize 1/2 → 0/1

def get_crops(h: int, w: int, crop_size: int, stride: int) -> list[tuple]:
    """Generate (row, col) top-left corners for all valid crops."""
    crops = []
    for r in range(0, h - crop_size + 1, stride):
        for c in range(0, w - crop_size + 1, stride):
            crops.append((r, c))
    # Ensure we always cover the bottom-right corner
    crops.append((h - crop_size, w - crop_size))
    return list(set(crops))

def _save_tif(arr: np.ndarray, path: Path):
    """Save (C, H, W) or (1, H, W) array as GeoTIFF."""
    c, h, w = arr.shape
    with rasterio.open(
        path, 'w',
        driver='GTiff',
        height=h, width=w,
        count=c,
        dtype=arr.dtype,
    ) as dst:
        dst.write(arr)

# preprocess_oscd(root="/scratch/htc/lhackel/data/OSCD", out_root="/scratch/htc/lhackel/data/OSCD_preprocessed", crop_size=224, train_stride=112)

### Features

In [9]:
def compute_normalization_stats(
    root: str | Path,
    bands: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    band_indices = [OSCD_BANDS.index(b) + 1 for b in bands]
    train_dir = Path(root) / 'train'
    per_file_means, per_file_vars, per_file_counts = [], [], []

    for sample_dir in sorted(train_dir.iterdir()):
        for fname in ('img1.tif', 'img2.tif'):
            with rasterio.open(sample_dir / fname) as src:
                img = src.read(band_indices).astype(np.float64)
            n = img.shape[1] * img.shape[2]
            per_file_means.append(img.mean(axis=(1, 2)))
            per_file_vars.append(img.var(axis=(1, 2)))
            per_file_counts.append(n)

    counts = np.array(per_file_counts)
    means  = np.array(per_file_means)
    vars_  = np.array(per_file_vars)
    total_n    = counts.sum()
    total_mean = (counts[:, None] * means).sum(axis=0) / total_n
    total_var  = (counts[:, None] * (vars_ + (means - total_mean) ** 2)).sum(axis=0) / total_n
    mean = total_mean.astype(np.float32)
    std  = np.sqrt(total_var).astype(np.float32)

    # Print ready-to-paste constant
    print("OSCD_NORM_STATS = {")
    for b, m, s in zip(bands, mean, std):
        print(f"    '{b}': ({m:.4f}, {s:.4f}),")
    print("}")

    return mean, std

# compute_normalization_stats(root, OSCD_BANDS)

In [10]:
from torch.utils.data import Dataset

OSCD_NORM_STATS = {
    'B01': (1592.9020, 281.0677),
    'B02': (1397.6360, 431.9554),
    'B03': (1328.5879, 558.9767),
    'B04': (1365.8481, 791.0626),
    'B05': (1491.9292, 743.7421),
    'B06': (1899.7970, 759.8222),
    'B07': (2115.0527, 838.9940),
    'B08': (2025.9111, 860.4077),
    'B8A': (2244.3398, 905.1946),
    'B09': (656.9305, 320.6088),
    'B10': (14.2359, 8.1941),
    'B11': (2023.4193, 1019.0800),
    'B12': (1487.6002, 875.3322),
}

class OSCDDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        split: str,
        bands: list[str],
        mean_std: tuple[np.ndarray, np.ndarray] | None = None,
        input_size: int = 224,
    ):
        self.root = Path(root)
        self.bands = [OSCD_BANDS.index(b) + 1 for b in bands]
        self.input_size = input_size
        self.samples = sorted((self.root / split).iterdir())

        if mean_std is not None:
            mean, std = mean_std
            self.mean = mean[:, None, None].astype(np.float32)  # (C, 1, 1) for broadcasting
            self.std  = std[:, None, None].astype(np.float32)
        else:
            self.mean = self.std = None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_dir = self.samples[idx]

        with rasterio.open(sample_dir / 'img1.tif') as src:
            img1 = src.read(self.bands).astype(np.float32)
        with rasterio.open(sample_dir / 'img2.tif') as src:
            img2 = src.read(self.bands).astype(np.float32)
        with rasterio.open(sample_dir / 'label.tif') as src:
            label = src.read(1)

        if self.mean is not None:
            img1 = (img1 - self.mean) / self.std
            img2 = (img2 - self.mean) / self.std

        img1  = self._resize(img1, 'bilinear')
        img2  = self._resize(img2, 'bilinear')
        label = self._resize(label[None].astype(np.float32), 'nearest')[0]

        return {
            'image1': torch.from_numpy(img1),
            'image2': torch.from_numpy(img2),
            'label':  torch.from_numpy(label).long(),
        }

    def _resize(self, arr: np.ndarray, mode: str) -> np.ndarray:
        if arr.shape[-1] == self.input_size:
            return arr
        t = torch.from_numpy(arr).unsqueeze(0)
        t = F.interpolate(t, size=(self.input_size, self.input_size), mode=mode,
                          align_corners=False if mode == 'bilinear' else None)
        return t.squeeze(0).numpy()
        
def get_mean_std(bands: list[str]) -> tuple[np.ndarray, np.ndarray]:
    mean = np.array([OSCD_NORM_STATS[b][0] for b in bands], dtype=np.float32)
    std  = np.array([OSCD_NORM_STATS[b][1] for b in bands], dtype=np.float32)
    return mean, std

root="/scratch/htc/lhackel/data/OSCD_preprocessed"

In [11]:
# All bands available in OSCD (imgs_X_rect)
OSCD_AVAILABLE_BANDS = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 
                         'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']

def get_oscd_bands_for_model(model_name: str) -> list[str]:
    """
    Select bands for OSCD by intersecting the model's translated_bands
    with what OSCD has available, preserving model band order.
    """
    translated = evaluated_models[model_name]["translated_bands"]
    return [b for b in translated if b in OSCD_AVAILABLE_BANDS]


# Check what each model gets
for name in ['prithvi2_300', 'prithvi2_600', 'dofa', 'dofa_large', 'terramind', 'terramind_large', 'ssl4eo_moco', 'ssl4eo_dino']:
    bands = get_oscd_bands_for_model(name)
    print(f"{name:>15}: {bands}")

   prithvi2_300: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
   prithvi2_600: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
           dofa: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
     dofa_large: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
      terramind: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
terramind_large: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
    ssl4eo_moco: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']
    ssl4eo_dino: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B10', 'B11', 'B12']


In [12]:
import warnings
import rasterio
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

def get_multiscale_features_and_labels_oscd(model, dl, ps, device):
    """
    Extract features from an OSCD dataloader returning {'image1', 'image2', 'label'}.
    
    Returns:
        features: (N, 2, 4, C, H_p, W_p)  - stacked t1/t2 features
        labels:   (N, H, W)
    """
    model.to(device)
    model.eval()

    all_features, all_labels = [], []

    for batch in tqdm(dl, desc="Extracting features", leave=False):
        img1 = batch['image1'].to(device)
        img2 = batch['image2'].to(device)
        y    = batch['label']

        with torch.no_grad():
            feat1 = select_four_layers(model(img1), img_size=224, patch_size=ps)
            feat2 = select_four_layers(model(img2), img_size=224, patch_size=ps)

        feat1 = F.normalize(feat1, dim=-1).cpu()
        feat2 = F.normalize(feat2, dim=-1).cpu()

        all_features.append(torch.stack([feat1, feat2], dim=1))  # (B, 2, 4, C, H_p, W_p)
        all_labels.append(y)

    return (
        torch.cat(all_features, dim=0).numpy(),   # (N, 2, 4, C, H_p, W_p)
        torch.cat(all_labels,   dim=0).numpy(),   # (N, H, W)
    )


def extract_multiscale_features_oscd(model, model_name, ps, base_path, root, device):
    """
    Extract and save multi-scale features for all OSCD splits and scales.
    """
    bands = get_oscd_bands_for_model(model_name)
    mean_std = get_mean_std(bands)

    for scale in tqdm(SCALES, desc="Iterating scales"):
        print(f"{model_name} : {scale}")

        try:
            model.backbone.set_to_scale(scale)
        except AttributeError:
            model.set_to_scale(scale)

        for split in ['train', 'test']:
            ds = OSCDDataset(root=root, split=split, bands=bands, mean_std=mean_std, input_size=224)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=16, pin_memory=True)

            features, labels = get_multiscale_features_and_labels_oscd(model, dl, ps, device)

            out_dir = Path(base_path) / model_name / f"scale_{scale}"
            out_dir.mkdir(parents=True, exist_ok=True)
            np.save(out_dir / f"features_oscd_{split}.npy", features)
            np.save(out_dir / f"labels_oscd_{split}.npy",   labels)

extract_cd_features = True
OSCD_ROOT = "/scratch/htc/lhackel/data/OSCD_preprocessed"

if extract_cd_features:
    for name_or_path in ['ssl4eo_moco', 'ssl4eo_dino']:
        ps = 14 if name_or_path == 'prithvi2_600' else 16
    
        backbone = BACKBONE_REGISTRY.build(**evaluated_models[name_or_path]["kwargs"])
        if "prithvi" in name_or_path or "dofa" in name_or_path or 'ssl4eo' in name_or_path:
            backbone = make_terratorch_prithvi_dofa_slimable(backbone)
        elif 'terramind' in name_or_path:
            backbone = make_terratorch_terramind_slimable(backbone)
    
        extract_multiscale_features_oscd(
            model=backbone,
            model_name=name_or_path,
            ps=ps,
            base_path=BASE_PATH,
            root=OSCD_ROOT,
            device=device,
        )

Loading weights


Iterating scales:   0%|                                            | 0/31 [00:00<?, ?it/s]

ssl4eo_moco : 0.001



Extracting features: 100%|████████████████████████████████| 11/11 [00:11<00:00,  2.75it/s]
                                                                                          
Iterating scales:   3%|█▏                                  | 1/31 [00:22<11:16, 22.53s/it]

ssl4eo_moco : 0.002


...


Extracting features: 100%|████████████████████████████████| 11/11 [00:05<00:00,  4.60it/s]
                                                                                          
Iterating scales:  94%|████████████████████████████████▋  | 29/31 [06:24<00:26, 13.09s/it]

ssl4eo_dino : 0.8



Extracting features: 100%|████████████████████████████████| 11/11 [00:05<00:00,  6.41it/s]
                                                                                          
Iterating scales:  97%|█████████████████████████████████▊ | 30/31 [06:37<00:13, 13.12s/it]

ssl4eo_dino : 1.0



Extracting features:  91%|█████████████████████████████   | 10/11 [00:05<00:00,  4.60it/s]
                                                                                          
Iterating scales: 100%|███████████████████████████████████| 31/31 [06:51<00:00, 13.26s/it]


### Eval

In [14]:
def train_binary(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    epochs: int = 30,
    lr: float = 0.01,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    weight_decay: float = 0.01,
    seed: int = 42
) -> dict:
    pl.seed_everything(seed)
    features_t = torch.from_numpy(features).float()
    labels_t = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    # Compute class weights from label distribution
    n_neg = (labels_t == 0).sum().item()
    n_pos = (labels_t == 1).sum().item()
    if n_pos > 0 and n_neg > 0:
        class_weights = torch.tensor([1.0, n_neg / n_pos], device=device)
    else:
        class_weights = None

    dataset = TensorDataset(features_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=8, persistent_workers=True)

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=255, weight=class_weights)

    history = {"loss": []}
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)

        scheduler.step()
        epoch_loss /= len(dataset)
        history["loss"].append(epoch_loss)
        print(f"  Training epoch {epoch+1:3d}/{epochs} | loss: {epoch_loss:.4f}")

    return history

from torchmetrics.classification import BinaryJaccardIndex, BinaryF1Score, BinaryAccuracy

def test_binary(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 32,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> dict:
    features_t = torch.from_numpy(features).float()
    labels_t   = torch.from_numpy(labels).long().squeeze(1)   # (N, H, W)

    dataset = TensorDataset(features_t, labels_t)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

    model = model.to(device)
    model.eval()

    iou_change  = BinaryJaccardIndex().to(device)   # IoU of change class (class 1)
    iou_nochange = BinaryJaccardIndex().to(device)  # IoU of no-change class (class 0)
    f1_metric   = BinaryF1Score().to(device)
    acc_metric  = BinaryAccuracy().to(device)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)           # (B, H, W)
            iou_change.update(preds, y)
            iou_nochange.update(1 - preds, 1 - y)   # flip for no-change class
            f1_metric.update(preds, y)
            acc_metric.update(preds, y)

    change_iou   = iou_change.compute().item()
    nochange_iou = iou_nochange.compute().item()
    metrics = {
        "mIoU":       (change_iou + nochange_iou) / 2,
        "change_iou":   change_iou,
        "nochange_iou": nochange_iou,
        "accuracy":   acc_metric.compute().item(),
        "macro_f1":   f1_metric.compute().item(),
    }
    print(f"  mIoU: {metrics['mIoU']:.4f} | Change IoU: {change_iou:.4f} | F1: {metrics['macro_f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    return metrics

import numpy as np
import pandas as pd

results = []

model_channel_data = [
    #('prithvi2_300', 1024, 16), 
    #('prithvi2_600', 1280, 14), 
    #('dofa', 768, 16), 
    #('dofa_large', 1024, 16), 
    #('terramind', 768, 16), 
    #('terramind_large', 1024, 16),
    ('ssl4eo_moco', 384, 16),
    # ('ssl4eo_dino', 384, 16),
]

eval_change_detection = True
if eval_change_detection:
    for name, d, p in model_channel_data:
        csv_path = f"{BASE_PATH}/{name}/change-detection_scores_probe_oscd.csv"
        for scale in SCALES:
            print(f"{name:>15} : {scale:5.3f}")
            print('-' * (15 + 3 + 5))

            head = SegmentationHead(in_channels=d, num_classes=2, embed_dim=128, img_size=224, patch_size=p)

            print("Loading training data")
            train_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_oscd_train.npy")  # (N, 2, 4, C, H_p, W_p)
            train_labels   = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_oscd_train.npy")    # (N, H, W)
            train_diff = train_features[:, 1] - train_features[:, 0]                               # (N, 4, C, H_p, W_p)

            train_binary(model=head, features=train_diff, labels=train_labels[:, None], epochs=30, lr=0.01, weight_decay=0.1)  # train_labels[:, None] to add a label dim

            print("Loading test data")
            test_features = np.load(f"{BASE_PATH}/{name}/scale_{scale}/features_oscd_test.npy")
            test_labels   = np.load(f"{BASE_PATH}/{name}/scale_{scale}/labels_oscd_test.npy")
            test_diff = test_features[:, 1] - test_features[:, 0]

            metrics = test_binary(model=head, features=test_diff, labels=test_labels[:, None])
            
            results.append({
                "model": name,
                "scale": scale,
                "dataset": "oscd",
                **metrics
            })

            df_row = pd.DataFrame(results[-1:])
            df_row.to_csv(csv_path, mode='a', index=False, header=not Path(csv_path).exists())
            print()

    print("Change detection complete")

    ssl4eo_moco : 0.001
-----------------------
Loading training data


[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.7081
  Training epoch   2/30 | loss: 0.5644
  Training epoch   3/30 | loss: 0.4901
  Training epoch   4/30 | loss: 0.4218
  Training epoch   5/30 | loss: 0.3543
  Training epoch   6/30 | loss: 0.3163
  Training epoch   7/30 | loss: 0.2853
  Training epoch   8/30 | loss: 0.2780
  Training epoch   9/30 | loss: 0.2498
  Training epoch  10/30 | loss: 0.2258
  Training epoch  11/30 | loss: 0.2113
  Training epoch  12/30 | loss: 0.2025
  Training epoch  13/30 | loss: 0.1840
  Training epoch  14/30 | loss: 0.1748
  Training epoch  15/30 | loss: 0.1657
  Training epoch  16/30 | loss: 0.1567
  Training epoch  17/30 | loss: 0.1497
  Training epoch  18/30 | loss: 0.1445
  Training epoch  19/30 | loss: 0.1408
  Training epoch  20/30 | loss: 0.1340
  Training epoch  21/30 | loss: 0.1295
  Training epoch  22/30 | loss: 0.1266
  Training epoch  23/30 | loss: 0.1221
  Training epoch  24/30 | loss: 0.1195
  Training epoch  25/30 | loss: 0.1186
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.7090
  Training epoch   2/30 | loss: 0.5710
  Training epoch   3/30 | loss: 0.4876
  Training epoch   4/30 | loss: 0.3875
  Training epoch   5/30 | loss: 0.3327
  Training epoch   6/30 | loss: 0.3017
  Training epoch   7/30 | loss: 0.2753
  Training epoch   8/30 | loss: 0.2490
  Training epoch   9/30 | loss: 0.2302
  Training epoch  10/30 | loss: 0.2132
  Training epoch  11/30 | loss: 0.1973
  Training epoch  12/30 | loss: 0.1920
  Training epoch  13/30 | loss: 0.1829
  Training epoch  14/30 | loss: 0.1687
  Training epoch  15/30 | loss: 0.1612
  Training epoch  16/30 | loss: 0.1487
  Training epoch  17/30 | loss: 0.1401
  Training epoch  18/30 | loss: 0.1341
  Training epoch  19/30 | loss: 0.1301
  Training epoch  20/30 | loss: 0.1248
  Training epoch  21/30 | loss: 0.1215
  Training epoch  22/30 | loss: 0.1190
  Training epoch  23/30 | loss: 0.1166
  Training epoch  24/30 | loss: 0.1147
  Training epoch  25/30 | loss: 0.1136
  Training epoch  26/30 |

...

  Training epoch   1/30 | loss: 0.6707
  Training epoch   2/30 | loss: 0.4735
  Training epoch   3/30 | loss: 0.3872
  Training epoch   4/30 | loss: 0.3272
  Training epoch   5/30 | loss: 0.2916
  Training epoch   6/30 | loss: 0.2471
  Training epoch   7/30 | loss: 0.2227
  Training epoch   8/30 | loss: 0.1957
  Training epoch   9/30 | loss: 0.1749
  Training epoch  10/30 | loss: 0.1621
  Training epoch  11/30 | loss: 0.1469
  Training epoch  12/30 | loss: 0.1398
  Training epoch  13/30 | loss: 0.1270
  Training epoch  14/30 | loss: 0.1203
  Training epoch  15/30 | loss: 0.1149
  Training epoch  16/30 | loss: 0.1112
  Training epoch  17/30 | loss: 0.1065
  Training epoch  18/30 | loss: 0.1034
  Training epoch  19/30 | loss: 0.1015
  Training epoch  20/30 | loss: 0.0991
  Training epoch  21/30 | loss: 0.0968
  Training epoch  22/30 | loss: 0.0939
  Training epoch  23/30 | loss: 0.0915
  Training epoch  24/30 | loss: 0.0900
  Training epoch  25/30 | loss: 0.0894
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6690
  Training epoch   2/30 | loss: 0.4667
  Training epoch   3/30 | loss: 0.4015
  Training epoch   4/30 | loss: 0.3336
  Training epoch   5/30 | loss: 0.2897
  Training epoch   6/30 | loss: 0.2590
  Training epoch   7/30 | loss: 0.2282
  Training epoch   8/30 | loss: 0.2002
  Training epoch   9/30 | loss: 0.1800
  Training epoch  10/30 | loss: 0.1704
  Training epoch  11/30 | loss: 0.1560
  Training epoch  12/30 | loss: 0.1450
  Training epoch  13/30 | loss: 0.1357
  Training epoch  14/30 | loss: 0.1282
  Training epoch  15/30 | loss: 0.1205
  Training epoch  16/30 | loss: 0.1181
  Training epoch  17/30 | loss: 0.1111
  Training epoch  18/30 | loss: 0.1062
  Training epoch  19/30 | loss: 0.1030
  Training epoch  20/30 | loss: 0.1009
  Training epoch  21/30 | loss: 0.0984
  Training epoch  22/30 | loss: 0.0964
  Training epoch  23/30 | loss: 0.0942
  Training epoch  24/30 | loss: 0.0930
  Training epoch  25/30 | loss: 0.0926
  Training epoch  26/30 |

[rank: 0] Seed set to 42


  Training epoch   1/30 | loss: 0.6639
  Training epoch   2/30 | loss: 0.4492
  Training epoch   3/30 | loss: 0.3745
  Training epoch   4/30 | loss: 0.3250
  Training epoch   5/30 | loss: 0.2727
  Training epoch   6/30 | loss: 0.2309
  Training epoch   7/30 | loss: 0.2062
  Training epoch   8/30 | loss: 0.1906
  Training epoch   9/30 | loss: 0.1702
  Training epoch  10/30 | loss: 0.1587
  Training epoch  11/30 | loss: 0.1482
  Training epoch  12/30 | loss: 0.1391
  Training epoch  13/30 | loss: 0.1382
  Training epoch  14/30 | loss: 0.1287
  Training epoch  15/30 | loss: 0.1193
  Training epoch  16/30 | loss: 0.1135
  Training epoch  17/30 | loss: 0.1096
  Training epoch  18/30 | loss: 0.1060
  Training epoch  19/30 | loss: 0.1033
  Training epoch  20/30 | loss: 0.1009
  Training epoch  21/30 | loss: 0.0979
  Training epoch  22/30 | loss: 0.0966
  Training epoch  23/30 | loss: 0.0936
  Training epoch  24/30 | loss: 0.0918
  Training epoch  25/30 | loss: 0.0914
  Training epoch  26/30 |

In [22]:
metric = 'macro_f1'
df = pd.read_csv(csv_path)
df = df[df.scale.isin(SCALES)]
df['max_val'] = df.groupby('model')[metric].transform('max')
df['rel_ret_rate'] = df[metric]/df['max_val']

df['rel_ret_rate_p'] = df['rel_ret_rate']*100
df['rel_ret_rate_p'] = df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
pivot = df.pivot(index='model', columns='scale', values='rel_ret_rate_p')
print(pivot.to_markdown())

| model           | 0.001   | 0.01   | 0.1    | 0.25   | 0.5   | 1.0    |
|:----------------|:--------|:-------|:-------|:-------|:------|:-------|
| dofa            | 97.3%   | 95.4%  | 100.0% | 94.9%  | 98.8% | 64.0%  |
| dofa_large      | 100.0%  | 98.6%  | 89.7%  | 99.6%  | 94.5% | 57.5%  |
| prithvi2_300    | 87.5%   | 89.5%  | 92.5%  | 100.0% | 96.3% | 78.3%  |
| prithvi2_600    | 88.9%   | 95.9%  | 100.0% | 87.4%  | 91.9% | 94.5%  |
| terramind       | 85.7%   | 85.5%  | 94.0%  | 70.7%  | 98.9% | 100.0% |
| terramind_large | 90.3%   | 80.1%  | 99.1%  | 99.0%  | 95.1% | 100.0% |


In [23]:
dataset = 'm-eurosat'
task = [t for t in geobench.task_iterator(benchmark_name="classification_v1.0") if t.dataset_name == dataset][0]
ds = task.get_dataset(split="train")
geobench_dataset_name = ds.task_specs.dataset_name
from geobench_evaluate import bandorders
bandnames = itemgetter(*terratorch_baseline_bandorders['dofa']["classification_v1.0"][geobench_dataset_name])(ds.task_specs.bands_info)
bandnames_param = [bn.name for bn in bandnames]
dm = task.get_pytorch_data_module(
    band_names=bandnames_param,
    train_transform=get_classification_transform(task, bandnames_param, train=True, input_size=224),
    eval_transform=get_classification_transform(task, bandnames_param, train=False, input_size=224),
    batch_size=128,
    num_workers=8,
)
batch = next(iter(dm.train_dataloader()))
batch['input'].shape, batch['label'].shape, ds.task_specs.label_type.n_classes

(torch.Size([128, 10, 224, 224]), torch.Size([128]), 10)

In [24]:
import torch
import torch.nn as nn
import timm
import lightning as L
from torchmetrics import Accuracy
from torchinfo import summary
torch.set_float32_matmul_precision('medium')

class GeoBenchClassifier(L.LightningModule):
    def __init__(
        self,
        model_name: str = "vit_tiny_patch16_224",
        num_classes: int = 10,
        in_chans: int = 10,
        img_size: int | None = 224,
        pretrained: bool = False,
        finetune: bool = True,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(
            model_name,
            pretrained=pretrained,
            in_chans=in_chans,
            num_classes=num_classes,
            **({'img_size': img_size} if img_size is not None else {})
        )
        if not finetune:
            for name, param in self.model.named_parameters():
                if "head" not in name:
                    param.requires_grad = False

        self.criterion = nn.CrossEntropyLoss()
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)
        img_size = 224 if img_size is None else img_size
        info = summary(self.model, input_size=(1, in_chans, img_size, img_size), verbose=0)
        self.model_flops_forward = info.total_mult_adds
        print(f"Model '{model_name}' has {self.model_flops_forward/1e9:.2f} GFLOPs.")
        self.model_name = model_name

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch["input"], batch["label"]
        return self.criterion(self(x), y)

    def test_step(self, batch, batch_idx):
        x, y = batch["input"], batch["label"]
        preds = self(x).argmax(dim=-1)
        acc = self.test_acc(preds, y)
        self.log("test/acc", self.test_acc, on_step=False, on_epoch=True)

    def on_test_epoch_end(self):
        self.test_result = self.test_acc.compute().item()

    def configure_optimizers(self):
        params = filter(lambda p: p.requires_grad, self.parameters())
        return torch.optim.AdamW(params, lr=self.hparams.lr, weight_decay=0.01)

In [25]:
epochs = 30
finetune, lr = False, 1e-3
# finetune, lr = True, 1e-4

In [26]:
class CachedDataset(torch.utils.data.Dataset):
    def __init__(self, batches):
        self.batches = batches
    def __len__(self):
        return len(self.batches)
    def __getitem__(self, idx):
        return self.batches[idx]

def cache_dataloader(dl):
    print("Caching...")
    batches = [batch for batch in dl]
    return torch.utils.data.DataLoader(
        CachedDataset(batches),
        batch_size=None,      # disables re-batching
        collate_fn=lambda x: x,  # return as-is
        num_workers=0,
        pin_memory=True
    )

cached_train = cache_dataloader(dm.train_dataloader())
cached_test = cache_dataloader(dm.test_dataloader())
print('Caching complete')

def run_classification(model_name, img_size = 224):
    trainer = L.pytorch.Trainer(max_epochs=epochs, accelerator="auto", enable_checkpointing=False, logger=False)

    model = GeoBenchClassifier(num_classes=10, in_chans=10, finetune=finetune, lr=lr, model_name=model_name, img_size=img_size)

    #trainer.fit(model, train_dataloaders=dm.train_dataloader())
    #trainer.test(model, dataloaders=dm.test_dataloader())
    trainer.fit(model, train_dataloaders=cached_train)
    trainer.test(model, dataloaders=cached_test)
    return model

Caching...
Caching...
Caching complete


In [27]:
recalculate_efficient_cv_models = False
if recalculate_efficient_cv_models:
    df_entries = {
        'name': [],
        'flops': [],
        'result': []
    }
    for model_name, img_size in [
        ('vit_tiny_patch16_224', 224),
        ('vit_small_patch16_224', 224),
        ('mobilenetv3_small_050', None),
        ('mobilenetv3_small_100', None),
        ('mobilenetv3_large_100', None),
        ('mobilevitv2_050', 224),
        ('mobilevitv2_100', 224),
        ('mobilevitv2_150', 224),
    ]:
        model = run_classification(model_name, img_size)
        df_entries['name'] += [model.model_name]
        df_entries['flops'] += [model.model_flops_forward]
        df_entries['result'] += [model.test_result]
    df_eff_cv = pd.DataFrame(df_entries)
    df_eff_cv.to_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/efficient_cv_models_{dataset}_{'ft' if finetune else 'linear'}.csv")

In [28]:
SCALES_FOR_EFFICIENT = [0.01, 1.0]
dfs = []
#for m in ['prithvi2_300', 'prithvi2_600', 'dofa', 'dofa_large', 'terramind', 'terramind_large']:
for m in ['prithvi2_300', 'dofa', 'terramind']:
    df = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/SlimAttn+SlimFFN/{m}/knn_scores_5way_{dataset}.csv").drop('Unnamed: 0', axis=1)
    df = df[df.dim==max(df.dim.values)].reset_index(drop=True)
    df = df.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
    df = df[df.index.isin(SCALES_FOR_EFFICIENT)]
    df['model'] = m
    df['max_val'] = max(df['accuracy'].values)
    df['rel_ret_rate'] = df['accuracy']/df['max_val']
    df['rel_ret_rate_p'] = df['rel_ret_rate']*100
    df['rel_ret_rate_p'] = df['rel_ret_rate_p'].apply(lambda x: f"{x:4.1f}%")
    df['metric_p'] = df['accuracy']*100
    df['metric_p'] = df['metric_p'].apply(lambda x: f"{x:4.1f}%")
    dfs.append(df.reset_index())  # reset so 'scale' becomes a column again

flop_table = pd.read_csv('/scratch/htc/lhackel/RSFM_redundancy/_features/model_flops.csv')

df = pd.concat(dfs, ignore_index=True)
df = df.merge(flop_table[['model', 'scale', 'flops_SASF']], on=['model', 'scale'], how='left')
df['model'] = df.apply(lambda row: f'{row["model"]} @ {row["scale"]*100: 5.1f}%', axis=1)
df['flops'] = df['flops_SASF'].astype(int)
df = df[['model', 'accuracy', 'flops']]

df_lin = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/efficient_cv_models_{dataset}_linear.csv")
df_lin['model'] = df_lin.apply(lambda row: f'{row["name"]} (lin)', axis=1)
df_lin['accuracy'] = df_lin['result']
df_ft = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/efficient_cv_models_{dataset}_ft.csv")
df_ft['model'] = df_ft.apply(lambda row: f'{row["name"]} (ft)', axis=1)
df_ft['accuracy'] = df_ft['result']

#df_joined = pd.concat([df, df_lin, df_ft], ignore_index=True).drop(columns=['name', 'result', 'Unnamed: 0'])
df_joined = pd.concat([df, df_ft], ignore_index=True).drop(columns=['name', 'result', 'Unnamed: 0'])
df_joined['GFLOPs'] = df_joined.flops.apply(lambda val: f"{val/1e9:.2f}")
print(df_joined[['model','accuracy','GFLOPs']].to_markdown(index=False))

| model                      |   accuracy |   GFLOPs |
|:---------------------------|-----------:|---------:|
| prithvi2_300 @   1.0%      |     0.7906 |     1.05 |
| prithvi2_300 @  100.0%     |     0.8306 |    59.85 |
| dofa @   1.0%              |     0.7136 |     0.93 |
| dofa @  100.0%             |     0.821  |    17.47 |
| terramind @   1.0%         |     0.736  |     0.69 |
| terramind @  100.0%        |     0.857  |    17.84 |
| vit_tiny_patch16_224 (ft)  |     0.858  |     0.1  |
| vit_small_patch16_224 (ft) |     0.884  |     0.21 |
| mobilenetv3_small_050 (ft) |     0.768  |     0.04 |
| mobilenetv3_small_100 (ft) |     0.747  |     0.07 |
| mobilenetv3_large_100 (ft) |     0.752  |     0.23 |
| mobilevitv2_050 (ft)       |     0.561  |     0.38 |
| mobilevitv2_100 (ft)       |     0.739  |     1.44 |
| mobilevitv2_150 (ft)       |     0.787  |     3.19 |


In [32]:
import numpy as np
from skdim.id import TwoNN

df_col = {
    'model': [],
    'scale': [],
    'Mean ID': [],
    'Std ID': [],
    'Min ID': [],
    'Max ID': [],
}

for model in ['SlimAttn+SlimFFN/prithvi2_600', 'SlimAttn+SlimFFN/dofa_large', 'SlimAttn+SlimFFN/terramind_large', 'timm_dinoV2_base', 'timm_mae_base']:
    for scale in [0.01, 1.0]:
        feats = np.load(f'/scratch/htc/lhackel/RSFM_redundancy/_features/{model}/scale_{scale}/features_m-eurosat_test.npy')
        lbls = np.load(f'/scratch/htc/lhackel/RSFM_redundancy/_features/{model}/scale_{scale}/labels_m-eurosat_test.npy')
        
        # --- Sanity check ---
        classes, counts = np.unique(lbls, return_counts=True)
        #print("Class distribution:")
        #for c, n in zip(classes, counts):
        #    print(f"  Class {c}: {n} samples")
        
        # --- Global ID ---
        id_global = TwoNN().fit(feats).dimension_
        #print(f"\nGlobal ID: {id_global:.2f}")
        
        # --- Per-class ID ---
        #print("\nPer-class ID:")
        ids = {}
        for cls in classes:
            X_cls = feats[lbls == cls]
            id_cls = TwoNN().fit(X_cls).dimension_
            ids[cls] = id_cls
            #print(f"  Class {cls}: ID = {id_cls:.2f}  (n={len(X_cls)})")
        
        # --- Summary stats ---
        id_values = list(ids.values())
        #print(f"\nMean per-class ID : {np.mean(id_values):.2f}")
        #print(f"Std  per-class ID : {np.std(id_values):.2f}")
        #print(f"Min  per-class ID : {np.min(id_values):.2f}")
        #print(f"Max  per-class ID : {np.max(id_values):.2f}")
        df_col['model'] += [f"{model} @ {scale*100}%"]
        df_col['scale'] += [scale]
        df_col['Mean ID'] += [np.mean(id_values)]
        df_col['Std ID'] += [np.std(id_values)]
        df_col['Min ID'] += [np.min(id_values)]
        df_col['Max ID'] += [np.max(id_values)]

df_ID = pd.DataFrame(df_col)
print(df_ID[['model', 'Mean ID', 'Std ID']].to_markdown(index=False))

| model                                     |   Mean ID |   Std ID |
|:------------------------------------------|----------:|---------:|
| SlimAttn+SlimFFN/prithvi2_600 @ 1.0%      |   6.1115  |  1.94907 |
| SlimAttn+SlimFFN/prithvi2_600 @ 100.0%    |   7.62244 |  2.13312 |
| SlimAttn+SlimFFN/dofa_large @ 1.0%        |   4.4297  |  1.12526 |
| SlimAttn+SlimFFN/dofa_large @ 100.0%      |  16.864   |  5.14958 |
| SlimAttn+SlimFFN/terramind_large @ 1.0%   |   4.07152 |  0.9143  |
| SlimAttn+SlimFFN/terramind_large @ 100.0% |   7.03483 |  1.78835 |
| timm_dinoV2_base @ 1.0%                   |   4.86747 |  1.8435  |
| timm_dinoV2_base @ 100.0%                 |  18.9126  |  3.76565 |
| timm_mae_base @ 1.0%                      |   4.28045 |  1.25518 |
| timm_mae_base @ 100.0%                    |   6.55448 |  1.89947 |


In [38]:
import numpy as np
from skdim.id import TwoNN

for cardinality in [10, 20, 50, 100]:
    df_col = {
        'model': [],
        'scale': [],
        'Mean ID': [],
        'Std ID': [],
        'Min ID': [],
        'Max ID': [],
    }
    
    for model in ['timm_dinoV2_base', 'timm_mae_base']:
        for scale in [0.01, 1.0]:
            feats = np.load(f'/scratch/htc/lhackel/RSFM_redundancy/_features/{model}/scale_{scale}/features_ImageNet{cardinality:03d}_test.npy')
            lbls = np.load(f'/scratch/htc/lhackel/RSFM_redundancy/_features/{model}/scale_{scale}/labels_ImageNet{cardinality:03d}_test.npy')
            lbls = lbls.squeeze(1)
            
            # --- Sanity check ---
            classes, counts = np.unique(lbls, return_counts=True)
            #print("Class distribution:")
            #for c, n in zip(classes, counts):
            #    print(f"  Class {c}: {n} samples")
            
            # --- Global ID ---
            id_global = TwoNN().fit(feats).dimension_
            #print(f"\nGlobal ID: {id_global:.2f}")
            
            # --- Per-class ID ---
            #print("\nPer-class ID:")
            ids = {}
            for cls in classes:
                X_cls = feats[lbls == cls]
                id_cls = TwoNN().fit(X_cls).dimension_
                ids[cls] = id_cls
                #print(f"  Class {cls}: ID = {id_cls:.2f}  (n={len(X_cls)})")
            
            # --- Summary stats ---
            id_values = list(ids.values())
            #print(f"\nMean per-class ID : {np.mean(id_values):.2f}")
            #print(f"Std  per-class ID : {np.std(id_values):.2f}")
            #print(f"Min  per-class ID : {np.min(id_values):.2f}")
            #print(f"Max  per-class ID : {np.max(id_values):.2f}")
            df_col['model'] += [f"{model} @ {scale*100}%"]
            df_col['scale'] += [scale]
            df_col['Mean ID'] += [np.mean(id_values)]
            df_col['Std ID'] += [np.std(id_values)]
            df_col['Min ID'] += [np.min(id_values)]
            df_col['Max ID'] += [np.max(id_values)]
    
    df_ID = pd.DataFrame(df_col)
    print(f"ImageNet n = {cardinality}")
    print(df_ID[['model', 'Mean ID', 'Std ID']].to_markdown(index=False))
    print()

ImageNet n = 10
| model                     |   Mean ID |   Std ID |
|:--------------------------|----------:|---------:|
| timm_dinoV2_base @ 1.0%   |   7.50316 |  1.77572 |
| timm_dinoV2_base @ 100.0% |  14.9568  |  2.18292 |
| timm_mae_base @ 1.0%      |   5.12382 |  1.17365 |
| timm_mae_base @ 100.0%    |   8.27696 |  1.88182 |

ImageNet n = 20
| model                     |   Mean ID |   Std ID |
|:--------------------------|----------:|---------:|
| timm_dinoV2_base @ 1.0%   |   7.57906 |  1.44994 |
| timm_dinoV2_base @ 100.0% |  15.0709  |  2.59053 |
| timm_mae_base @ 1.0%      |   5.13354 |  1.03474 |
| timm_mae_base @ 100.0%    |   8.31661 |  1.6617  |

ImageNet n = 50
| model                     |   Mean ID |   Std ID |
|:--------------------------|----------:|---------:|
| timm_dinoV2_base @ 1.0%   |   7.60632 | 1.36088  |
| timm_dinoV2_base @ 100.0% |  14.863   | 3.81802  |
| timm_mae_base @ 1.0%      |   5.2596  | 0.962085 |
| timm_mae_base @ 100.0%    |   7.91727 | 1.56592

In [ ]:
import pandas as pd
import geobench
from tqdm import tqdm
import redundant_feature_helpers
from scripts.redundant_features import extract_features, calculate_result
from slimrsfms.helper import SlimStrategy

In [ ]:
model_name

In [21]:
#model, max_dim = 'terramind', 768
#model, max_dim = 'dofa', 768
model, max_dim = 'prithvi2_300', 1024
dataset = 'm-eurosat'

bs = 128
image_size = 224

task = [t for t in geobench.task_iterator(benchmark_name="classification_v1.0") if t.dataset_name == dataset][0]
ds = task.get_dataset(split="train")

get_features = True
knn = True

# SlimStrategy.FIRST, SlimStrategy.LAST, or SlimStrategy.RANDOM
# seed only relevant for SlimStrategy.RANDOM

variants = []
variants += [('_first', SlimStrategy.FIRST, 0)]
variants += [('_last', SlimStrategy.LAST, 0)]
for i in range(20):
    variants += [(f'_random:seed{i}', SlimStrategy.RANDOM, i)]

model, dm, model_name = redundant_feature_helpers.load_model(model, geobench_dataset=ds, geobench_task=task, batch_size=bs, image_size=image_size)
for i, (modification, strategy, seed) in enumerate(variants):
    print(f"Running variant {i+1}/{len(variants)}: {modification} with strategy {strategy} and seed {seed} for {model_name}")
    if get_features:
        extract_features(model=model, dm=dm, model_name=f'implicit_order/{model_name}{modification}', dataset_name=dataset, strategy=strategy, seed=seed)
        print("Extraction Complete")
    if knn:
        calculate_result(model_name=f'implicit_order/{model_name}{modification}', dataset_name=dataset, max_dim=max_dim, downstream_type='knn')
        print("KNN Complete")
    print("Done\n")

Running variant 1/22: _first with strategy SlimStrategy.FIRST and seed 0 for prithvi2_300


Iterating scales: 100%|██████████████████████████████████████████████████████████████████████| 31/31 [07:22<00:00, 14.27s/it]


Extraction Complete


Calculating KNN: 100%|███████████████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.84it/s]


KNN Complete
Done

Running variant 2/22: _last with strategy SlimStrategy.LAST and seed 0 for prithvi2_300


...

KNN Complete
Done

Running variant 22/22: _random:seed19 with strategy SlimStrategy.RANDOM and seed 19 for prithvi2_300


Iterating scales: 100%|██████████████████████████████████████████████████████████████████████| 31/31 [07:31<00:00, 14.56s/it]


Extraction Complete


Calculating KNN: 100%|███████████████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.79it/s]

KNN Complete
Done



In [6]:
SCALES = [0.001, 0.01, 0.1, 0.25, 0.5, 1.0]

dfs = []
metric = 'accuracy'

for version in [
    'terramind_first', 
    'terramind_last'
] + [f'terramind_random:seed{i}' for i in range(20)]:
    df_part = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/implicit_order/{version}/knn_scores_5way_m-eurosat.csv").drop('Unnamed: 0', axis=1)
    df_part = df_part[df_part.dim==768].reset_index(drop=True)
    df_part = df_part.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
    df_part = df_part[df_part.index.isin(SCALES)]
    df_part['version'] = version.split('_')[1]
    dfs.append(df_part.reset_index())


df_part = pd.read_csv(f"/scratch/htc/lhackel/RSFM_redundancy/_features/SlimAttn+SlimFFN/terramind/knn_scores_5way_m-eurosat.csv").drop('Unnamed: 0', axis=1)
df_part = df_part[df_part.dim==768].reset_index(drop=True)
df_part = df_part.groupby('scale').mean().drop(['seed', 'dim'], axis=1)
df_part = df_part[df_part.index.isin(SCALES)]
df_part['version'] = 'CONTROL (should be same as "first")'
dfs.append(df_part.reset_index())
    
df = pd.concat(dfs, ignore_index=True)
pivot = df.pivot(index='version', columns='scale', values=metric)

# exclude CONTROL for summary
non_control = pivot.loc[pivot.index != 'CONTROL (should be same as "first")']

for name, func in [('mean', 'mean'), ('std', 'std'), ('min', 'min'), ('max', 'max')]:
    pivot.loc[name] = getattr(non_control, func)()

print(pivot.loc[non_control.index].to_markdown())
print()  # blank line
print("Summary:")
summary = pd.DataFrame({name: getattr(non_control, func)() for name, func in [('mean', 'mean'), ('std', 'std'), ('min', 'min'), ('max', 'max')]}).T
summary.index.name = 'stat'
print(summary.to_markdown())

| version       |   0.001 |   0.01 |   0.1 |   0.25 |   0.5 |   1.0 |
|:--------------|--------:|-------:|------:|-------:|------:|------:|
| first         |   0.735 |  0.736 | 0.778 |  0.814 | 0.851 | 0.857 |
| last          |   0.712 |  0.728 | 0.779 |  0.816 | 0.813 | 0.857 |
| random:seed0  |   0.732 |  0.74  | 0.777 |  0.82  | 0.851 | 0.857 |
| random:seed1  |   0.726 |  0.735 | 0.767 |  0.822 | 0.844 | 0.857 |
| random:seed10 |   0.722 |  0.726 | 0.763 |  0.797 | 0.835 | 0.857 |
| random:seed11 |   0.715 |  0.725 | 0.776 |  0.803 | 0.823 | 0.857 |
| random:seed12 |   0.728 |  0.732 | 0.737 |  0.791 | 0.833 | 0.857 |
| random:seed13 |   0.726 |  0.727 | 0.776 |  0.813 | 0.843 | 0.857 |
| random:seed14 |   0.722 |  0.731 | 0.77  |  0.818 | 0.845 | 0.857 |
| random:seed15 |   0.723 |  0.728 | 0.797 |  0.801 | 0.826 | 0.857 |
| random:seed16 |   0.73  |  0.736 | 0.776 |  0.804 | 0.811 | 0.857 |
| random:seed17 |   0.723 |  0.727 | 0.789 |  0.796 | 0.821 | 0.857 |
| random:seed18 |   

In [8]:
# Build summary with first, last, random (mean ± std)
random_versions = [idx for idx in non_control.index if idx.startswith('random')]
first_row = pivot.loc['first']
last_row = pivot.loc['last']
random_subset = pivot.loc[random_versions]

summary_rows = []
for label, data in [('first', first_row.to_frame().T), ('last', last_row.to_frame().T), ('random (n=20)', random_subset)]:
    row = {}
    for scale in SCALES:
        m = data[scale].mean() * 100
        s = data[scale].std() * 100
        if len(data) == 1:
            row[scale] = f"{m:.1f}%"
        else:
            row[scale] = f"{m:.1f}% ({s:.2f}%)"
    summary_rows.append(pd.Series(row, name=label))

summary_df = pd.DataFrame(summary_rows)
summary_df.index.name = 'version'
print(summary_df.to_markdown())

| version       | 0.001         | 0.01          | 0.1           | 0.25          | 0.5           | 1.0           |
|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|
| first         | 73.5%         | 73.6%         | 77.8%         | 81.4%         | 85.1%         | 85.7%         |
| last          | 71.2%         | 72.8%         | 77.9%         | 81.6%         | 81.3%         | 85.7%         |
| random (n=20) | 72.6% (0.51%) | 73.2% (0.52%) | 77.4% (1.42%) | 81.0% (0.97%) | 83.1% (1.30%) | 85.7% (0.00%) |


In [1]:
import io
from pathlib import Path
import timm
import torch
from lightning.pytorch import LightningDataModule
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from scripts.redundant_features import extract_features, calculate_result
from scripts.terratorch_slimable import make_terratorch_prithvi_dofa_slimable

In [5]:
class CIFAR10DataModule(LightningDataModule):
    def __init__(self, data_dir: str, transform=None, bs: int = 256, num_workers: int = 8):
        super().__init__()
        self.data_dir = data_dir
        self.transform = transform
        self.bs = bs
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = CIFAR10(
            root=self.data_dir,
            train=True,
            download=True,
            transform=self.transform,
        )
        self.val_dataset = CIFAR10(
            root=self.data_dir,
            train=False,
            download=True,
            transform=self.transform,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.bs,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.bs,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def test_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.bs,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

In [11]:
extract = False
evaluate = False

if extract:
    print(">>> Extracting CIFAR-10")
    model = timm.create_model(
        "vit_base_patch16_224.mae",
        #"vit_base_patch14_dinov2", img_size=224,
        pretrained=True,
    )
    model = make_terratorch_prithvi_dofa_slimable(model)

    config = resolve_data_config({'input_size': (3, 224, 224)}, model=model)
    transform = create_transform(**config)

    dm = CIFAR10DataModule(
        data_dir="/scratch/htc/lhackel/data/cifar10/",
        transform=transform,
        bs=256,
        num_workers=8,
    )
    dm.setup(None)

    extract_features(model, dm, "timm_mae_base", "CIFAR10")
    #extract_features(model, dm, "timm_dinoV2_base", "CIFAR10")

if evaluate:
    print(">>> Calculating Results on CIFAR-10")
    calculate_result(
        model_name="timm_mae_base",
        #model_name="timm_dinoV2_base",
        dataset_name="CIFAR10",
        max_dim=768,
        downstream_type='knn',
    )

print("Done")

>>> Extracting CIFAR-10


Iterating scales:   0%|                                             | 0/31 [00:00<?, ?it/s]

Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)


Iterating scales:   3%|█▏                                   | 1/31 [00:29<14:45, 29.51s/it]

Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)


...

Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)
Procedure unknown for forward, assuming model(x)


Iterating scales: 100%|████████████████████████████████████| 31/31 [18:07<00:00, 35.06s/it]


>>> Calculating Results on CIFAR-10


Calculating KNN: 100%|█████████████████████████████████████| 93/93 [09:09<00:00,  5.91s/it]

Done
